# 🎯 Trenowanie Wake Word "Hey Hermes" z openWakeWord

**Cel:** Wytrenować model wake word detection dla frazy **"Hey Hermes"**
używając [openWakeWord](https://github.com/dscripka/openWakeWord) —
lekkiego, lokalnego detektora wake word działającego na CPU.

## Jak to działa

openWakeWord pozwala na trenowanie własnych wake wordów przez:
1. **Generowanie syntetycznych danych** — augmentacja nagrań frazy przez szumy,
   zmiany tempa, wysokości dźwięku
2. **Trenowanie modelu** — lightweight CNN na cechach Mel-frequency
3. **Eksport do ONNX/TFLite** — do użycia na urządzeniach mobilnych, Raspberry Pi, Home Assistant

## Wymagane zbiory danych
- Nagrania frazy "Hey Hermes" (minimum 20-50 nagrań, różne osoby/głosy)
- Nagrania tła / szumów (opcjonalnie, do augmentacji)
- Nagrania "negative" — inne frazy, rozmowy (opcjonalnie)

## Wynik końcowy
- Plik `hey_hermes.onnx` — model do wnioskowania w Python
- Plik `hey_hermes.tflite` — model do Home Assistant / urządzeń mobilnych
- Plik `hey_hermes_config.yaml` — konfiguracja dla openWakeWord w Home Assistant


In [ ]:
# =============================================================================
# Komórka 2: Instalacja zależności
# =============================================================================

# --- Instalacja openWakeWord i zależności ---
!pip install -q openwakeword
!pip install -q numpy librosa soundfile pydub
!pip install -q onnx onnxruntime  # dla eksportu
!pip install -q tensorflow-cpu     # dla eksportu TFLite (opcjonalnie)
!pip install -q matplotlib tqdm    # do wizualizacji

# --- Weryfikacja ---
import openwakeword
print(f"openWakeWord wersja: {openwakeword.__version__}")

import librosa
import numpy as np
import soundfile as sf
from pathlib import Path
from tqdm.notebook import tqdm
print("✅ Wszystkie zależności zainstalowane.")

In [ ]:
# =============================================================================
# Komórka 3: Generowanie danych treningowych
# =============================================================================

import os
import librosa
import numpy as np
import soundfile as sf
from pathlib import Path
import random
from google.colab import files
import IPython.display as ipd

# --- Foldery ---
DATA_DIR = Path("/content/wakeword_data")
POSITIVE_DIR = DATA_DIR / "positive"  # Nagrania "Hey Hermes"
NEGATIVE_DIR = DATA_DIR / "negative"  # Inne frazy (opcjonalnie)
BACKGROUND_DIR = DATA_DIR / "background"  # Szumy tła
OUTPUT_DIR = DATA_DIR / "augmented"

for d in [POSITIVE_DIR, NEGATIVE_DIR, BACKGROUND_DIR, OUTPUT_DIR]:
    d.mkdir(parents=True, exist_ok=True)

print("📁 Foldery utworzone.")
print(f"  Pozytywne ("Hey Hermes"): {POSITIVE_DIR}")
print(f"  Negatywne:                {NEGATIVE_DIR}")
print(f"  Tło/szumy:                {BACKGROUND_DIR}")
print(f"  Augmentowane:              {OUTPUT_DIR}")
print()

# --- Opcja A: Nagraj własne próbki ---
print("=" * 60)
print("🎤 NAGRANIE PRÓBEK 'Hey Hermes'")
print("=" * 60)
print("Uruchom poniższe nagrywanie. Powiedz 'Hey Hermes' wyraźnie.")
print("Nagraj minimum 20-50 próbek (różne osoby, różne intonacje).")
print()

# --- Funkcja do nagrywania ---
from IPython.display import HTML, Javascript

def record_audio(duration=3, sample_rate=16000):
    """Nagraj audio z mikrofonu w Colab"""
    from google.colab import output
    import base64
    import json

    display(HTML(f'''
    <script>
    async function record() {{
        const stream = await navigator.mediaDevices.getUserMedia({{ audio: true }});
        const recorder = new MediaRecorder(stream);
        const chunks = [];
        recorder.ondataavailable = e => chunks.push(e.data);
        recorder.start();
        await new Promise(r => setTimeout(r, {duration * 1000}));
        recorder.stop();
        await new Promise(r => recorder.onstop = r);
        const blob = new Blob(chunks, {{ type: 'audio/webm' }});
        const reader = new FileReader();
        reader.readAsDataURL(blob);
        reader.onloadend = () => {{
            google.colab.kernel.invokeFunction('notebook.save_audio', [reader.result], {{}});
        }};
    }}
    record();
    </script>
    <button onclick="record()">🎤 Nagraj (szybki klik)</button>
    '''))

    # Odbieranie audio
    recording_id = len(list(POSITIVE_DIR.glob("*.wav")))

    def save_audio(data_url):
        import base64
        audio_bytes = base64.b64decode(data_url.split(',')[1])
        with open(POSITIVE_DIR / f"hey_hermes_{recording_id:03d}.webm", "wb") as f:
            f.write(audio_bytes)
        # Konwersja do WAV (16000 Hz, mono)
        import subprocess
        subprocess.run([
            "ffmpeg", "-y",
            "-i", str(POSITIVE_DIR / f"hey_hermes_{recording_id:03d}.webm"),
            "-ar", "16000",
            "-ac", "1",
            str(POSITIVE_DIR / f"hey_hermes_{recording_id:03d}.wav")
        ], capture_output=True)
        (POSITIVE_DIR / f"hey_hermes_{recording_id:03d}.webm").unlink(missing_ok=True)
        print(f"✅ Zapisano: hey_hermes_{recording_id:03d}.wav")

    output.register_callback("notebook.save_audio", save_audio)

# Uruchom nagrywanie
record_audio(duration=2)  # 2 sekundy na próbkę

In [ ]:
# =============================================================================
# Komórka 4: Trenowanie modelu openWakeWord
# =============================================================================

import numpy as np
import librosa
from pathlib import Path
from sklearn.model_selection import train_test_split
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
import warnings
warnings.filterwarnings('ignore')

# --- Parametry ---
SAMPLE_RATE = 16000
N_MFCC = 13
MAX_LEN = 128  # ramki czasowe MFCC
EPOCHS = 50
BATCH_SIZE = 32
LEARNING_RATE = 0.001

POSITIVE_DIR = Path("/content/wakeword_data/positive")
NEGATIVE_DIR = Path("/content/wakeword_data/negative")

def extract_features(file_path):
    """Wyodrębnij cechy MFCC z pliku audio"""
    try:
        audio, sr = librosa.load(file_path, sr=SAMPLE_RATE, mono=True)
        # Normalizacja głośności
        audio = audio / (np.max(np.abs(audio)) + 1e-8)
        # MFCC
        mfcc = librosa.feature.mfcc(y=audio, sr=sr, n_mfcc=N_MFCC)
        # Padding/trim do MAX_LEN
        if mfcc.shape[1] < MAX_LEN:
            pad_width = MAX_LEN - mfcc.shape[1]
            mfcc = np.pad(mfcc, ((0, 0), (0, pad_width)), mode='constant')
        else:
            mfcc = mfcc[:, :MAX_LEN]
        return mfcc.T  # (time, mfcc_features)
    except Exception as e:
        print(f"  ⚠️ Błąd przetwarzania {file_path}: {e}")
        return None

# --- Wczytywanie danych ---
print("📂 Wczytywanie danych treningowych...")

features = []
labels = []

# Pozytywne (Hey Hermes -> 1)
pos_files = list(POSITIVE_DIR.glob("*.wav"))
print(f"  Pozytywne: {len(pos_files)} plików")
for f in pos_files:
    feat = extract_features(f)
    if feat is not None:
        features.append(feat)
        labels.append(1)

# Negatywne (inne frazy -> 0)
neg_files = list(NEGATIVE_DIR.glob("*.wav"))
print(f"  Negatywne: {len(neg_files)} plików")
for f in neg_files:
    feat = extract_features(f)
    if feat is not None:
        features.append(feat)
        labels.append(0)

# Jeśli brak negatywnych, wygeneruj szum jako negatywne
if len(neg_files) == 0:
    print("  ⚠️ Brak nagrań negatywnych. Generuję szum jako dane negatywne...")
    for _ in range(len(pos_files)):
        noise = np.random.randn(MAX_LEN, N_MFCC) * 0.1
        features.append(noise)
        labels.append(0)

X = np.array(features)
y = np.array(labels)

print(f"\n📊 Kształt danych: {X.shape}")
print(f"📊 Etykiety: {y.sum()} pozytywnych, {len(y)-y.sum()} negatywnych")

# --- Podział na trening/validację ---
X_train, X_val, y_train, y_val = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
print(f"  Trening: {len(X_train)} próbek")
print(f"  Walidacja: {len(X_val)} próbek")

# --- Model ---
print("\n🧠 Budowanie modelu...")

model = keras.Sequential([
    layers.Input(shape=(MAX_LEN, N_MFCC)),
    layers.Conv1D(64, 3, activation='relu', padding='same'),
    layers.BatchNormalization(),
    layers.MaxPooling1D(2),
    layers.Conv1D(128, 3, activation='relu', padding='same'),
    layers.BatchNormalization(),
    layers.MaxPooling1D(2),
    layers.Conv1D(256, 3, activation='relu', padding='same'),
    layers.BatchNormalization(),
    layers.GlobalAveragePooling1D(),
    layers.Dropout(0.3),
    layers.Dense(128, activation='relu'),
    layers.Dropout(0.3),
    layers.Dense(1, activation='sigmoid')
])

model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=LEARNING_RATE),
    loss='binary_crossentropy',
    metrics=['accuracy', keras.metrics.Precision(), keras.metrics.Recall()]
)

model.summary()

# --- Callbacks ---
callbacks = [
    keras.callbacks.EarlyStopping(
        monitor='val_loss', patience=10, restore_best_weights=True
    ),
    keras.callbacks.ReduceLROnPlateau(
        monitor='val_loss', factor=0.5, patience=5, min_lr=1e-6
    )
]

# --- Trenowanie ---
print("\n🏋️ Rozpoczęcie trenowania...")
history = model.fit(
    X_train, y_train,
    validation_data=(X_val, y_val),
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    callbacks=callbacks,
    verbose=1
)

print("\n✅ Trenowanie zakończone!")

# --- Wizualizacja ---
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# Loss
axes[0].plot(history.history['loss'], label='Trening')
axes[0].plot(history.history['val_loss'], label='Walidacja')
axes[0].set_title('Loss')
axes[0].set_xlabel('Epoka')
axes[0].set_ylabel('Loss')
axes[0].legend()
axes[0].grid(True)

# Accuracy
axes[1].plot(history.history['accuracy'], label='Trening')
axes[1].plot(history.history['val_accuracy'], label='Walidacja')
axes[1].set_title('Accuracy')
axes[1].set_xlabel('Epoka')
axes[1].set_ylabel('Accuracy')
axes[1].legend()
axes[1].grid(True)

# Precision/Recall
axes[2].plot(history.history['precision'], label='Precision')
axes[2].plot(history.history['recall'], label='Recall')
axes[2].set_title('Precision & Recall')
axes[2].set_xlabel('Epoka')
axes[2].set_ylabel('Wartość')
axes[2].legend()
axes[2].grid(True)

plt.tight_layout()
plt.savefig('/content/wakeword_data/training_history.png', dpi=150)
plt.show()

# --- Ewaluacja końcowa ---
loss, acc, prec, rec = model.evaluate(X_val, y_val, verbose=0)
print(f"\n📊 Wyniki na zbiorze walidacyjnym:")
print(f"  Accuracy:  {acc:.4f}")
print(f"  Precision: {prec:.4f}")
print(f"  Recall:    {rec:.4f}")

In [ ]:
# =============================================================================
# Komórka 5: Eksport modelu do ONNX i TFLite
# =============================================================================

import numpy as np
import tensorflow as tf
from pathlib import Path

OUTPUT_DIR = Path("/content/wakeword_data")
MODEL_SAVE_PATH = OUTPUT_DIR / "hey_hermes_model.keras"

# Zapisz model w formacie Keras
model.save(MODEL_SAVE_PATH)
print(f"✅ Model zapisany: {MODEL_SAVE_PATH}")

# --- Eksport do TFLite ---
print("\n📦 Eksport do TFLite...")

# Konwersja do TFLite z optymalizacją
converter = tf.lite.TFLiteConverter.from_keras_model(model)
converter.optimizations = [tf.lite.Optimize.DEFAULT]
converter.target_spec.supported_types = [tf.float16]

tflite_model = converter.convert()

tflite_path = OUTPUT_DIR / "hey_hermes.tflite"
with open(tflite_path, "wb") as f:
    f.write(tflite_model)

tflite_size = tflite_path.stat().st_size / 1024
print(f"✅ TFLite zapisany: {tflite_path}")
print(f"   Rozmiar: {tflite_size:.1f} KB")

# --- Eksport do ONNX ---
print("\n📦 Eksport do ONNX...")

try:
    import tf2onnx
    import onnx

    onnx_path = OUTPUT_DIR / "hey_hermes.onnx"

    # Specyfikacja wejścia
    spec = (tf.TensorSpec((None, MAX_LEN, N_MFCC), tf.float32, name="input"),)

    # Konwersja
    output_path = tf2onnx.convert.from_keras(
        model,
        input_signature=spec,
        opset=14,
        output_path=str(onnx_path)
    )

    onnx_size = onnx_path.stat().st_size / 1024
    print(f"✅ ONNX zapisany: {onnx_path}")
    print(f"   Rozmiar: {onnx_size:.1f} KB")

    # Weryfikacja modelu ONNX
    onnx_model = onnx.load(onnx_path)
    onnx.checker.check_model(onnx_model)
    print("   Model ONNX przeszedł walidację.")

except ImportError:
    print("⚠️ tf2onnx nie jest zainstalowany. Pomijam eksport ONNX.")
    print("   Zainstaluj: pip install tf2onnx onnx")

# --- Test inferencji TFLite ---
print("\n🔬 Test inferencji z TFLite...")

interpreter = tf.lite.Interpreter(model_path=str(tflite_path))
interpreter.allocate_tensors()

input_details = interpreter.get_input_details()
output_details = interpreter.get_output_details()

# Test z losowymi danymi
test_input = np.random.randn(1, MAX_LEN, N_MFCC).astype(np.float32)
interpreter.set_tensor(input_details[0]['index'], test_input)
interpreter.invoke()
result = interpreter.get_tensor(output_details[0]['index'])

print(f"   Wejście: {test_input.shape}")
print(f"   Wyjście: {result} (prawdopodobieństwo wykrycia)")
print("✅ Inferencja TFLite działa poprawnie.")

# --- Podsumowanie ---
print("\n" + "=" * 50)
print("📁 Wygenerowane pliki:")
for f in sorted(OUTPUT_DIR.glob("hey_hermes*")):
    size_kb = f.stat().st_size / 1024
    print(f"  • {f.name}  ({size_kb:.1f} KB)")
print("=" * 50)

# Komórka 6: Instrukcje wdrożenia

Po wytrenowaniu modelu, masz dwa pliki do wdrożenia:

---

## 📱 Opcja A: Home Assistant (TFLite)

1. **Pobierz plik** `hey_hermes.tflite` z Colab:
   ```python
   from google.colab import files
   files.download('/content/wakeword_data/hey_hermes.tflite')
   ```

2. **Skopiuj do Home Assistant:**
   ```bash
   # Na maszynie z Home Assistant:
   mkdir -p /config/wakewords/
   cp hey_hermes.tflite /config/wakewords/
   ```

3. **Skonfiguruj openWakeWord w config:**
   ```yaml
   # configuration.yaml
   openwakeword:
     models:
       - wake_word_hey_hermes: /config/wakewords/hey_hermes.tflite
   ```

4. **Zrestartuj Home Assistant**

5. **Skonfiguruj Assist Pipeline:**
   - Settings → Voice Assistants → Assist → Add Pipeline
   - Wake word: `Hey Hermes`
   - STT: Wyoming Whisper
   - Conversation agent: Hermes
   - TTS: Wyoming Piper / ElevenLabs

---

## 🖥️ Opcja B: Standalone Python (ONNX)

```python
import onnxruntime as ort
import numpy as np
import librosa

# Wczytaj model ONNX
session = ort.InferenceSession("hey_hermes.onnx")
input_name = session.get_inputs()[0].name

# Funkcja detekcji
def detect_wakeword(audio_path, threshold=0.5):
    audio, sr = librosa.load(audio_path, sr=16000, mono=True)
    audio = audio / (np.max(np.abs(audio)) + 1e-8)
    mfcc = librosa.feature.mfcc(y=audio, sr=sr, n_mfcc=13)
    
    # Przygotuj input (sliding window co 128 ramek)
    window_size = 128
    scores = []
    for i in range(0, mfcc.shape[1], window_size // 2):
        chunk = mfcc[:, i:i+window_size]
        if chunk.shape[1] < window_size:
            chunk = np.pad(chunk, ((0,0),(0,window_size-chunk.shape[1])))
        input_data = chunk.T.astype(np.float32).reshape(1, window_size, 13)
        score = session.run(None, {input_name: input_data})[0][0][0]
        scores.append(score)
    
    return max(scores) > threshold

# Test
if detect_wakeword("test_hey_hermes.wav"):
    print("🔊 Wykryto 'Hey Hermes'!")
else:
    print("❌ Nie wykryto")
```

---

## 📋 Lista kontrolna wdrożenia

- [ ] Model wytrenowany (accuracy > 0.95, precision > 0.9)
- [ ] Plik TFLite skopiowany do Home Assistant
- [ ] `openwakeword:` dodany do `configuration.yaml`
- [ ] Assist Pipeline skonfigurowany z wake wordem
- [ ] STT/TTS działają (testowane)
- [ ] Conversation agent wskazuje na Hermes API
- [ ] Przetestowane na telefonie przez Tailscale

---

## ⚠️ Wskazówki

- **Jakość nagrań:** Nagraj próbki w cichym pomieszczeniu, blisko mikrofonu
- **Różnorodność:** Nagraj różne osoby, różne intonacje (pytanie, rozkaz, neutralnie)
- **False positives:** Jeśli model zbyt często się wybudza, zwiększ `threshold` (np. 0.7)
- **False negatives:** Jeśli nie wykrywa, zmniejsz `threshold` (np. 0.3) lub dodaj więcej danych
- **CPU vs GPU:** openWakeWord działa na CPU — model < 1 MB jest idealny
- **Tailscale:** Upewnij się że Tailscale działa na telefonie przed testem wake worda
